# BERT 모델 예측 및 성능 분석 (Recall Analysis)

HuggingFace Hub에 업로드된 학습 완료 KLUE-BERT 모델을 불러와,
test_set_80.xlsx에 대해 예측을 수행하고 클래스별 성능을 상세 분석합니다.

## 목적
- 학습된 BERT 모델의 최종 성능을 독립적으로 검증
- 7개 피드백 레벨별 Precision, Recall, F1 Score 확인
- 혼동 행렬(Confusion Matrix)로 오분류 패턴 파악
- 예측 확률과 함께 결과를 Excel로 저장

## 사전학습 모델(Pre-trained Model) 활용
HuggingFace Hub에서 미세조정(Fine-tuning)이 완료된 모델을 다운로드합니다.
토크나이저와 모델 가중치가 함께 저장되어 있어, 별도의 학습 없이
바로 예측에 사용할 수 있습니다.

### 라이브러리 임포트

PyTorch, HuggingFace Transformers, scikit-learn의 평가 지표를 불러옵니다.

In [4]:
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, f1_score, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

## 1. 설정

### 설정값 정의

HuggingFace 저장소 경로, 데이터 경로, 모델 하이퍼파라미터를 설정합니다.
read-only 토큰으로 모델을 다운로드합니다.

In [ ]:
import os

# HuggingFace 설정
HF_REPO = "StudioLS2/bert_feedback"
HF_TOKEN = os.environ.get("HF_TOKEN", "")  # 환경변수에서 토큰 로드 (export HF_TOKEN=hf_xxx)

# 데이터 경로 (seed42 테스트셋 사용)
DATA_PATH = "../data/feedback_data.xlsx"

# 모델 설정
MAX_LENGTH = 128
BATCH_SIZE = 32

# 디바이스 설정
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Device: {device}")

## 2. 모델 및 토크나이저 로드

### 모델 로드

HuggingFace Hub에서 학습 완료된 KLUE-BERT 모델과 토크나이저를 다운로드합니다.
- `AutoTokenizer`: 텍스트를 토큰(숫자)으로 변환하는 토크나이저
- `AutoModelForSequenceClassification`: 7클래스 분류용 BERT 모델
- `model.eval()`: 평가 모드로 전환 (학습하지 않고 예측만 수행)

In [6]:
from huggingface_hub import login
login(token=HF_TOKEN)

tokenizer = AutoTokenizer.from_pretrained(HF_REPO)
model = AutoModelForSequenceClassification.from_pretrained(HF_REPO)
model = model.to(device)
model.eval()  # 평가 모드: Dropout 비활성화, 결정적 예측

print(f"Model: {model.config.architectures}")
print(f"Num labels: {model.config.num_labels}")
print(f"Vocab size: {model.config.vocab_size}")

tokenizer.json:   0%|          | 0.00/752k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Model: ['BertForSequenceClassification']
Num labels: 7
Vocab size: 32000


## 3. 테스트 데이터 로드

### 테스트 데이터 로드

test_set_80.xlsx에서 피드백 텍스트와 정답 레이블을 로드합니다.
각 레벨(0~6)의 샘플 수(분포)를 확인합니다.

In [7]:
all_df = pd.read_excel(DATA_PATH)
df_test = all_df[all_df['split_seed42'] == 'test'].reset_index(drop=True)
print(f"Test set: {len(df_test)} samples")
print(f"Columns: {df_test.columns.tolist()}")
print(f"\nLabel distribution:")
print(df_test['label'].value_counts().sort_index())
df_test.head()

## 4. Dataset 및 DataLoader

### Dataset 클래스 및 DataLoader 생성

텍스트를 BERT 토크나이저로 변환하고, DataLoader로 배치 처리합니다.
- 토크나이저: 텍스트 → 토큰 ID + 어텐션 마스크
- DataLoader: 데이터를 batch_size(32)개씩 묶어 모델에 전달

In [8]:
class FeedbackDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length, labels=None):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

test_dataset = FeedbackDataset(
    texts=df_test['feedback_text'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    labels=df_test['label'].values
)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"DataLoader: {len(test_loader)} batches")

DataLoader: 13 batches


## 5. 예측 수행

### 예측 수행

`model.eval()` 모드에서 `torch.no_grad()` 컨텍스트 내에서 예측합니다.
- `model.eval()`: Dropout, BatchNorm 등을 비활성화하여 결정적 예측
- `torch.no_grad()`: 기울기 계산을 건너뛰어 메모리 절약 및 속도 향상
- `softmax`: 모델 출력(logits)을 0~1 사이의 확률로 변환

In [9]:
all_preds = []   # 예측 레이블 저장
all_labels = []  # 실제 정답 레이블 저장
all_probs = []   # 각 클래스별 확률 저장

with torch.no_grad():  # 기울기 계산 비활성화 (메모리 절약, 속도 향상)
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label']

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)  # 순전파(Forward) 실행
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

print(f"Predictions: {len(all_preds)}")
print(f"Accuracy: {(all_preds == all_labels).mean():.4f}")

Predictions: 416
Accuracy: 0.9495


## 6. 평가 결과

### Classification Report (클래스별 성능 보고서)

각 피드백 레벨(0~6)별로 다음 지표를 계산합니다:
- **Precision (정밀도)**: 모델이 해당 레벨로 예측한 것 중 실제로 맞는 비율
- **Recall (재현율)**: 실제 해당 레벨인 것 중 모델이 올바르게 찾아낸 비율
- **F1 Score**: Precision과 Recall의 조화평균
- **Support**: 각 레벨의 실제 샘플 수

Recall이 낮은 클래스 = 모델이 잘 찾지 못하는 피드백 유형입니다.

In [10]:
# Classification Report
label_names = [f"Level {i}" for i in range(7)]
print("=== Classification Report ===")
print(classification_report(all_labels, all_preds, target_names=label_names, digits=4))

# Key metrics
macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')
kappa = cohen_kappa_score(all_labels, all_preds)
accuracy = (all_preds == all_labels).mean()

print(f"\n=== Summary ===")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print(f"Cohen's κ:   {kappa:.4f}")

=== Classification Report ===
              precision    recall  f1-score   support

     Level 0     1.0000    0.9167    0.9565        12
     Level 1     0.9800    0.9515    0.9655       103
     Level 2     1.0000    0.9825    0.9912        57
     Level 3     0.9732    0.9932    0.9831       146
     Level 4     0.7619    0.8889    0.8205        36
     Level 5     0.9216    0.8545    0.8868        55
     Level 6     0.8571    0.8571    0.8571         7

    accuracy                         0.9495       416
   macro avg     0.9277    0.9206    0.9230       416
weighted avg     0.9522    0.9495    0.9501       416


=== Summary ===
Accuracy:    0.9495
Macro F1:    0.9230
Weighted F1: 0.9501
Cohen's κ:   0.9345


### 혼동 행렬 (Confusion Matrix)

실제 레벨(행) vs 예측 레벨(열)을 격자로 표시합니다.
- 대각선: 정확한 예측 (값이 클수록 좋음)
- 비대각선: 오분류 (어떤 레벨끼리 혼동되는지 파악)

예를 들어, Level 2와 Level 3이 자주 혼동된다면,
이 두 레벨의 코딩 기준을 더 명확히 해야 할 수 있습니다.

In [11]:
# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
print("=== Confusion Matrix ===")
print(pd.DataFrame(cm, index=label_names, columns=label_names))

=== Confusion Matrix ===
         Level 0  Level 1  Level 2  Level 3  Level 4  Level 5  Level 6
Level 0       11        1        0        0        0        0        0
Level 1        0       98        0        2        3        0        0
Level 2        0        1       56        0        0        0        0
Level 3        0        0        0      145        0        1        0
Level 4        0        0        0        0       32        3        1
Level 5        0        0        0        2        6       47        0
Level 6        0        0        0        0        1        0        6


## 7. 예측 결과 저장

### 예측 결과 저장

예측값, 정답 여부, 각 클래스별 예측 확률을 포함한 상세 결과를 Excel로 저장합니다.
확률값을 통해 모델이 얼마나 확신을 가지고 예측했는지 분석할 수 있습니다.

In [12]:
# 예측 결과 DataFrame 생성
df_result = df_test.copy()
df_result['predicted'] = all_preds
df_result['correct'] = (all_preds == all_labels)  # 정답 여부 (True/False)

# 각 클래스별 확률 추가
for i in range(7):  # 7개 클래스 각각의 예측 확률을 별도 열로 저장
    df_result[f'prob_{i}'] = all_probs[:, i]

# 저장
output_path = "./bert_prediction_results.xlsx"
df_result.to_excel(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Total: {len(df_result)}, Correct: {df_result['correct'].sum()}, Wrong: {(~df_result['correct']).sum()}")
df_result.head(10)